<a href="https://colab.research.google.com/github/abhay-joshi007/customer-churn-prediction/blob/main/Customer_churn_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Data loaded successfully:")
print(df.head())

Data loaded successfully:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingM

In [3]:
# Check for missing values and types
print(df.info())

# TotalCharges is 'object' type, let's convert it to numeric
# 'coerce' will turn any errors (like empty strings) into 'NaN' (Not a Number)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill the few missing values with 0 (e.g., for new customers with 0 tenure)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Drop the customerID column as it's not a predictive feature
df = df.drop('customerID', axis=1)

print("\nData cleaned. Missing values in TotalCharges handled.")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [6]:
# Check for missing values and types
print(df.info())

# TotalCharges is 'object' type, let's convert it to numeric
# 'coerce' will turn any errors (like empty strings) into 'NaN' (Not a Number)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill the few missing values with 0 (e.g., for new customers with 0 tenure)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print("\nData cleaned. Missing values in TotalCharges handled.")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [7]:
# Identify numerical and categorical columns
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_cols = df.drop(columns=numeric_cols + ['Churn']).columns

# --- Preprocessing ---

# 1. Process Categorical Data using One-Hot Encoding
# This creates new columns for each category (e.g., 'PaymentMethod_Credit Card')
df_processed = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# 2. Process the Target Variable ('Churn')
# Convert 'Yes'/'No' to 1/0
df_processed['Churn'] = df_processed['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

# 3. Scale Numerical Data
scaler = StandardScaler()
df_processed[numeric_cols] = scaler.fit_transform(df_processed[numeric_cols])

print("Data has been fully preprocessed:")
print(df_processed.head())

Data has been fully preprocessed:
     tenure  MonthlyCharges  TotalCharges  Churn  gender_Male  \
0 -1.277445       -1.160323     -0.992611      0        False   
1  0.066327       -0.259629     -0.172165      0         True   
2 -1.236724       -0.362660     -0.958066      1         True   
3  0.514251       -0.746535     -0.193672      0         True   
4 -1.236724        0.197365     -0.938874      1        False   

   SeniorCitizen_1  Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0            False         True           False             False   
1            False        False           False              True   
2            False        False           False              True   
3            False        False           False             False   
4            False        False           False              True   

   MultipleLines_No phone service  ...  StreamingTV_No internet service  \
0                            True  ...                            False   
1         

In [8]:
# 'y' is the column we want to predict
y = df_processed['Churn']

# 'X' is all the other columns
X = df_processed.drop('Churn', axis=1)

# Split the data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (5634, 30)
Testing data shape: (1409, 30)


In [9]:
# --- Model 1: Logistic Regression ---
print("\n--- Training Logistic Regression ---")
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

# Evaluate Logistic Regression
y_pred_log = log_model.predict(X_test)
print("--- Logistic Regression Results ---")
print(classification_report(y_test, y_pred_log))


# --- Model 2: Random Forest ---
print("\n--- Training Random Forest ---")
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate Random Forest
y_pred_rf = rf_model.predict(X_test)
print("--- Random Forest Results ---")
print(classification_report(y_test, y_pred_rf))



--- Training Logistic Regression ---
--- Logistic Regression Results ---
              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409


--- Training Random Forest ---
--- Random Forest Results ---
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1035
           1       0.64      0.49      0.56       374

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

